# Aula 1, Word2Vec

Notebook executável que acompanha a aula [01-word2vec.md](../../lessons/modulo-04-word-embeddings/01-word2vec.md).

## O que vamos fazer aqui

Vamos treinar um skip-gram do zero, com numpy, sobre um pequeno corpus em que gato e
cachorro vivem em contextos quase idênticos, comendo, bebendo e dormindo. A
expectativa é ver os vetores de gato e cachorro terminarem próximos, enquanto gato e
peixe, que não dividem contexto, ficam distantes.

É um modelo minúsculo, mas suficiente para ver os embeddings emergirem de verdade,
sem nenhuma biblioteca além do numpy.

## Preparando o corpus e os pares (centro, contexto)

O skip-gram aprende a prever as palavras de contexto a partir da palavra central.
Então o primeiro passo é, para cada palavra de cada frase, gerar os pares com as
palavras vizinhas dentro de uma janela de duas posições.

In [ ]:
import numpy as np
import re

frases = [
    "o gato come peixe", "o cachorro come carne",
    "o gato bebe leite", "o cachorro bebe agua",
    "o gato dorme no sofa", "o cachorro dorme no tapete",
    "a crianca gosta do gato", "a crianca gosta do cachorro",
]


def tokenizar(texto):
    return re.findall(r"\w+", texto.lower())


tokens_frases = [tokenizar(f) for f in frases]
vocab = sorted({w for fr in tokens_frases for w in fr})
vi = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

pares = []
for fr in tokens_frases:
    for i, centro in enumerate(fr):
        for j in range(max(0, i - 2), min(len(fr), i + 3)):
            if j != i:
                pares.append((vi[centro], vi[fr[j]]))

print("vocabulário:", V, "palavras")
print("pares de treino:", len(pares))

Temos um vocabulário pequeno e algumas dezenas de pares. Cada par diz: quando
vejo esta palavra central, esta outra costuma aparecer por perto. É só disso que o
modelo precisa para aprender.

## Treinando o skip-gram

Cada palavra central vira um embedding (uma linha de `W1`). A partir dele, calculamos
pontuações para todo o vocabulário, passamos pela softmax e ajustamos os vetores para
tornar as palavras de contexto corretas mais prováveis. É gradiente descendente, igual
ao do Módulo 2, agora sobre vetores de palavra.

In [ ]:
rng = np.random.default_rng(0)
D = 10                                  # dimensão dos embeddings
W1 = rng.normal(0, 0.1, (V, D))         # embeddings de entrada (o que nos interessa)
W2 = rng.normal(0, 0.1, (D, V))         # vetores de saída
taxa = 0.05

for epoca in range(300):
    rng.shuffle(pares)
    for centro, contexto in pares:
        h = W1[centro]                  # embedding da palavra central
        z = h @ W2                      # pontuações para todo o vocabulário
        z -= z.max()                    # estabiliza a exponencial
        p = np.exp(z); p /= p.sum()     # softmax
        p[contexto] -= 1                # gradiente da entropia cruzada
        grad_h = W2 @ p                 # gradiente do embedding, usando o W2 atual
        W2 -= taxa * np.outer(h, p)     # atualiza os vetores de saída
        W1[centro] -= taxa * grad_h     # atualiza o embedding central

print("treino concluído, embeddings de dimensão", D)

Depois de 300 passadas pelos pares, as linhas de `W1` deixaram de ser ruído
aleatório e passaram a codificar relações de contexto. Vamos medir essas relações.

## Medindo a similaridade semântica

A similaridade do cosseno entre dois embeddings diz o quanto eles apontam na mesma
direção. Vamos comparar pares que esperamos próximos (gato e cachorro, come e bebe)
com um par que esperamos distante (gato e peixe).

In [ ]:
def similaridade(a, b):
    va, vb = W1[vi[a]], W1[vi[b]]
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb)))


print("gato ~ cachorro:", round(similaridade("gato", "cachorro"), 3))
print("come ~ bebe    :", round(similaridade("come", "bebe"), 3))
print("gato ~ peixe   :", round(similaridade("gato", "peixe"), 3))

Com a semente fixa, gato e cachorro ficam com similaridade alta, por volta de 0,55,
assim como come e bebe, enquanto gato e peixe ficam perto de zero ou negativo. Se você
trocar a semente ou os hiperparâmetros, os números mudam, mas o padrão é firme:
palavras que vivem nos mesmos contextos ganham vetores parecidos. Isso é o que o Bag
of Words não conseguia. Para o projeto, liste os vizinhos mais próximos de uma palavra
pela similaridade do cosseno.